# ex05 · 数值稳定性与初始化（对应教材 4.8）

> **做题流程**：先预测现象，再运行观察，最后补全正确初始化。
> **做完再看** `solutions/ex05-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节搞清楚「梯度消失/爆炸」和「初始化」这两个训练深网络的关键问题（面试高频）。

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from torch import nn

## 题 1 🌱 梯度消失（sigmoid 饱和）

先预测：sigmoid 在 x 很大或很小时，它的梯度（导数）会怎样？运行看曲线。

In [ ]:
x = torch.arange(-8.0, 8.0, 0.1, requires_grad=True)
y = torch.sigmoid(x)
y.backward(torch.ones_like(x))
plt.figure(figsize=(6, 3))
plt.plot(x.detach().numpy(), y.detach().numpy(), label='sigmoid')
plt.plot(x.detach().numpy(), x.grad.numpy(), label='gradient')
plt.legend()
plt.title('sigmoid 及其梯度：两端梯度趋近 0')
plt.show()

## 题 2 🔧 梯度爆炸（矩阵连乘）

先预测：一个 4×4 随机矩阵，反复和随机矩阵相乘 100 次，数值会怎样？运行观察。

In [ ]:
torch.manual_seed(0)
M = torch.normal(0, 1, size=(4, 4))
print('初始最大绝对值:', round(float(M.abs().max()), 3))
for i in range(100):
    M = torch.mm(M, torch.normal(0, 1, size=(4, 4)))
print('连乘 100 次后最大绝对值:', float(M.abs().max()))

## 题 3 🔧 找 bug：两种错误初始化

运行下面两段，观察现象，说出**原因**（面试高频「为什么不能全 0 / 过大初始化」）。

In [ ]:
# Bug A：全 0 初始化 → 隐藏层梯度全 0，永远学不动
torch.manual_seed(0)
W1 = torch.zeros(6, 6, requires_grad=True)
b1 = torch.zeros(6, requires_grad=True)
W2 = torch.randn(6, 1, requires_grad=True) * 0.01
x = torch.randn(4, 6)
y = torch.randn(4, 1)
H = torch.relu(x @ W1 + b1)
out = H @ W2
l = ((out - y) ** 2).mean()
l.backward()
print('Bug A（全 0 初始化）: W1 的梯度是否全为 0 →', bool((W1.grad == 0).all().item()))
print('  → 隐藏层收不到任何梯度，训练不动')

# Bug B：过大初始化 → 激活饱和
torch.manual_seed(1)
w_big = torch.normal(0, 10, size=(50, 8))
x2 = torch.randn(1, 50)
out2 = torch.sigmoid(x2 @ w_big)
print('Bug B（过大初始化）: sigmoid 输出范围 →', round(float(out2.min()), 4), '~', round(float(out2.max()), 4))
print('  → 输出几乎全是 0 或 1，梯度趋近 0')

## 题 4 🔧 正确初始化（TODO 5.1）

补全一个正确的初始化函数。先预测：为什么小的随机值（如 std=0.01）能避免上面的两个问题？

In [ ]:
def init_weights(m):
    # TODO 5.1: 对 nn.Linear 层用小的随机值初始化（nn.init.normal_(m.weight, std=0.01)）
    raise NotImplementedError('⚠ TODO 5.1: 正确初始化未完成')

In [ ]:
try:
    net5 = nn.Linear(10, 10)
    init_weights(net5)
    assert not bool((net5.weight == 0).all().item()), '权重不应全 0'
    print('✓ 初始化后权重方差:', round(float(net5.weight.var()), 6), '（很小的正数）')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

## 题 5 🔧 Xavier 初始化（面试高频）

Xavier 根据层宽自动缩放初始值，让前向/反向的方差都大致不变。先预测：Xavier 初始化后权重的方差大约是多少？运行对照理论值。

In [ ]:
net = nn.Sequential(nn.Linear(20, 20), nn.Linear(20, 20))
for m in net:
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
print('Xavier 初始化后第一层权重方差:', round(float(net[0].weight.var()), 6))
print('理论值 2/(fan_in+fan_out) =', round(2 / (20 + 20), 6))

## 小结与面试衔接

- 梯度消失：饱和激活（sigmoid）两端导数趋近 0，连乘后梯度消失
- 梯度爆炸：数值连乘指数放大（4×4 矩阵连乘 100 次直接爆炸）
- 不能全 0 初始化：隐藏层输出全相同 → 梯度全 0（或对称，各神经元学得一样）
- 不能过大初始化：激活进入饱和区 → 梯度消失
- Xavier：按层宽缩放方差，缓解梯度消失/爆炸；一轮「底层运行机制 + 骨架组件」考点